# Getting started with krovlab

`roof` turns a building footprint and a pitch into a roof you can read
quantities off: faces, hips, ridges, valleys, heights, covering area.

Each example below prints the quantities, then draws:

- **plan** — brown **walls** (the building), grey **eave** (where the roof
  ends), skeleton over both
- **3D** — the solid, with the same wall line at height 0

With no overhang the two lines sit on top of each other. With an
overhang you see the roof project past the walls.

A `Failure` has no solid; you get the input outline instead.

For plans that go wrong, see `limitations.ipynb`.


In [ ]:
from collections import defaultdict

import plotly.graph_objects as go

from krovlab import Failure, Roof, roof, topology_hash
from krovlab.viz import plan_view, solid_view, wavefront_steps, wavefront_view


def describe(result: Roof | Failure) -> None:
    if isinstance(result, Failure):
        print(f"Failure  kind={result.kind}\n  {result.reason}")
        return
    by_kind: dict[str, list[float]] = defaultdict(list)
    for arc in result.arcs:
        by_kind[arc.kind].append(arc.length)
    print(f"terrain:           {result.validity.is_terrain}")
    print(f"ridge height:      {result.ridge_height:.3f} m")
    print(f"total sloped area: {result.total_sloped_area:.3f} m²")
    print(f"nodes {len(result.nodes)}, faces {len(result.faces)}, arcs {len(result.arcs)}")
    for kind, lengths in by_kind.items():
        print(f"  {kind:7s}  {len(lengths)} run(s)  total {sum(lengths):.3f} m")
    if not result.validity.is_terrain:
        for reason in result.validity.reasons:
            print(f"  !! {reason}")


def footprint_outline(
    footprint: list[tuple[float, float]],
    holes: list[list[tuple[float, float]]] | None = None,
    title: str = "",
) -> go.Figure:
    """Plan of the input rings — used when there is no roof to draw."""
    fig = go.Figure()
    if footprint:
        xs = [p[0] for p in footprint] + [footprint[0][0]]
        ys = [p[1] for p in footprint] + [footprint[0][1]]
        fig.add_trace(
            go.Scatter(
                x=xs, y=ys, mode="lines+markers", name="walls",
                line={"color": "#7c2d12", "width": 2},
                fill="toself", fillcolor="rgba(124, 45, 18, 0.10)",
                marker={"size": 7, "color": "#7c2d12"},
            )
        )
    for i, hole in enumerate(holes or []):
        if not hole:
            continue
        xs = [p[0] for p in hole] + [hole[0][0]]
        ys = [p[1] for p in hole] + [hole[0][1]]
        fig.add_trace(
            go.Scatter(
                x=xs, y=ys, mode="lines+markers",
                name=f"hole {i}" if len(holes or []) > 1 else "hole",
                line={"color": "#1d4ed8", "width": 2, "dash": "dot"},
                marker={"size": 6, "color": "#1d4ed8"},
            )
        )
    fig.update_layout(
        title=title, xaxis_title="x (m)", yaxis_title="y (m)",
        yaxis_scaleanchor="x", yaxis_scaleratio=1,
        template="plotly_white", height=420, legend_title="input",
    )
    return fig


def show(
    title: str,
    result: Roof | Failure,
    footprint: list[tuple[float, float]] | None = None,
    holes: list[list[tuple[float, float]]] | None = None,
) -> None:
    """Print quantities, then draw.

    A valid roof: plan + 3D. Brown = walls; grey eave = where the roof
    ends. They coincide with no overhang; with an overhang the eaves
    sit outside the walls.
    A constructed but non-terrain roof: plan only.
    A Failure: the input outline.
    """
    print(title)
    describe(result)
    if isinstance(result, Roof) and result.validity.is_terrain:
        plan = plan_view(result, walls=footprint, wall_holes=holes)
        plan.update_layout(title=f"{title} — plan (brown = walls, eave = roof edge)")
        plan.show()
        solid = solid_view(result, walls=footprint, wall_holes=holes)
        solid.update_layout(title=f"{title} — 3D (brown = walls at height 0)")
        solid.show()
    elif isinstance(result, Roof):
        plan = plan_view(result, walls=footprint, wall_holes=holes)
        plan.update_layout(title=f"{title} — plan (not a terrain)")
        plan.show()
    elif footprint is not None:
        footprint_outline(footprint, holes, title=f"{title} — {result.kind}").show()


## A square hip roof

A 10 m square at 45° is the smallest interesting case. The four hips
meet at an apex whose height is half the side times `tan(pitch)`:
`5 × tan(45°) = 5 m`.


In [ ]:
square = [(0.0, 0.0), (10.0, 0.0), (10.0, 10.0), (0.0, 10.0)]
square_roof = roof(square, 45.0)
assert isinstance(square_roof, Roof)
show("10 m square at 45°", square_roof, square)


### Faces

Each face rises from one footprint edge. `plan_area` is the horizontal
projection; `sloped_area` is what covering is bought by
(`plan_area / cos(pitch)`).


In [ ]:
print(f"{'edge':>4}  {'pitch':>6}  {'plan m²':>8}  {'sloped m²':>10}")
for face in square_roof.faces:
    print(
        f"{face.edge_index:4d}  {face.pitch:6.1f}  "
        f"{face.plan_area:8.3f}  {face.sloped_area:10.3f}"
    )


## A rectangle: one ridge

A 10 × 6 m rectangle at 45° has a ridge of length `10 − 6 = 4 m` at
height 3 m.


In [ ]:
rectangle = [(0.0, 0.0), (10.0, 0.0), (10.0, 6.0), (0.0, 6.0)]
rect_roof = roof(rectangle, 45.0)
assert isinstance(rect_roof, Roof)
show("10 × 6 m rectangle at 45°", rect_roof, rectangle)


## Pitch spellings

`100%` is `1:1` is `45°`. US `4:12` is `atan(4/12) ≈ 18.4°`.


In [ ]:
for spelling in (45.0, (1, 1), "1:1", "100%", "4:12"):
    result = roof(square, spelling)
    assert isinstance(result, Roof)
    print(f"{spelling!r:>8}  ridge {result.ridge_height:7.3f} m  pitch {result.faces[0].pitch:.3f}°")

shallow = roof(square, "4:12")
assert isinstance(shallow, Roof)
show("10 m square at 4:12", shallow, square)


## A different pitch per wall


In [ ]:
mixed = roof(square, [60.0, 45.0, 60.0, 45.0])
assert isinstance(mixed, Roof)
show("square, pitches [60, 45, 60, 45]", mixed, square)


## L and U: valleys


In [ ]:
l_shape = [
    (0.0, 0.0), (10.0, 0.0), (10.0, 6.0),
    (3.0, 6.0), (3.0, 10.0), (0.0, 10.0),
]
u_shape = [
    (0.0, 0.0), (10.0, 0.0), (10.0, 8.0), (7.0, 8.0),
    (7.0, 3.0), (3.0, 3.0), (3.0, 8.0), (0.0, 8.0),
]
show("L-shape at 45°", roof(l_shape, 45.0), l_shape)
show("U-shape at 45°", roof(u_shape, 45.0), u_shape)


## A courtyard

A hole is a second ring at **eave height**. The roof does not cover it.
A single pitch covers the hole edges too — you do not pass a separate
pitch for the courtyard unless you want one.

A 10 m square with a centred 4 m courtyard at 45°: ridge 1.5 m, plan
area 84 m².


In [ ]:
courtyard = [(3.0, 3.0), (7.0, 3.0), (7.0, 7.0), (3.0, 7.0)]
court_roof = roof(square, 45.0, holes=[courtyard])
assert isinstance(court_roof, Roof)
show("square with 4 m courtyard at 45°", court_roof, square, [courtyard])
print("faces (outer 0–3, hole 4–7):", [(f.edge_index, f.pitch) for f in court_roof.faces])


## Can a hole be a circle?

Not as a true curve — every wall is a straight segment. Approximate a
circle with an n-gon. You get **one inward face per segment**, not a
cone of revolution. More sides look rounder and cost more faces.


In [ ]:
import math

def circle(cx: float, cy: float, radius: float, n: int = 16) -> list[tuple[float, float]]:
    return [
        (cx + radius * math.cos(2 * math.pi * i / n),
         cy + radius * math.sin(2 * math.pi * i / n))
        for i in range(n)
    ]

round_well = circle(5.0, 5.0, 1.5, n=16)
round_roof = roof(square, 45.0, holes=[round_well])
assert isinstance(round_roof, Roof)
show("16-sided circular courtyard, r = 1.5 m", round_roof, square, [round_well])
print(f"{len(round_roof.faces)} faces: 4 outer + 16 on the hole")


## A hole without its own pitch — and why that is not a chimney

Two different requests get mixed up here.

**"I don't want to specify a pitch for the hole."** A scalar pitch
already applies to every edge, including the hole. The courtyard above
used `roof(square, 45, holes=[...])` — no list.

**"A cut-through, like a chimney."** A chimney is a hole *through the
slope*, above the eaves. This library's hole is a courtyard at eave
height: the roof still grows inward faces down to that opening. A
chimney, dormer or rooflight is a penetration, and is not in the model.

The closest thing you *can* do is gable every hole edge (`pitch = 90`):
vertical inner walls, no inward-sloping faces. That is a light well
with vertical sides, still opening at eave height, still not a chimney.


In [ ]:
# Outer 45°, hole 90° — vertical courtyard walls.
well_pitches = [45.0, 45.0, 45.0, 45.0, 90.0, 90.0, 90.0, 90.0]
well = roof(square, well_pitches, holes=[courtyard])
assert isinstance(well, Roof)
show("courtyard with vertical inner walls (not a chimney)", well, square, [courtyard])
print("faces only on outer edges:", [f.edge_index for f in well.faces])


## Gable ends and a shed

`pitch = 90` on an outer edge is a gable. Three gables leave a shed.
Gabling every outer edge cannot close.


In [ ]:
show("rectangle, east wall gabled", roof(rectangle, [45.0, 90.0, 45.0, 45.0]), rectangle)
show("rectangle, both short sides gabled", roof(rectangle, [45.0, 90.0, 45.0, 90.0]), rectangle)
show("rectangle shed (three gables)", roof(rectangle, [45.0, 90.0, 90.0, 90.0]), rectangle)
show("every edge a gable", roof(square, 90.0), square)


## Eaves overhang — walls vs roof edge

`overhang` is metres **past the walls**. The library offsets the
footprint outward (and each hole inward) and roofs that larger polygon.

In the drawings:

- **brown `walls`** — the building you passed in
- **grey `eave`** — where the roof ends
- the band between them is the overhang

On a 10 × 6 m rectangle with 0.5 m overhang the eaves run from
(−0.5, −0.5) to (10.5, 6.5). A courtyard hole shrinks: a 4 m well
becomes 3 m. An overhang that closes a hole or folds a thin wing is a
named `Failure`.


In [ ]:
overhung = roof(rectangle, 45.0, overhang=0.5)
assert isinstance(overhung, Roof)
show("rectangle, 0.5 m overhang — eaves outside the walls", overhung, rectangle)

eaves = [a for a in overhung.arcs if a.kind == "eave"]
print("eave corners (roof edge):")
for arc in eaves:
    n = overhung.nodes[arc.start]
    print(f"  ({n.x:.2f}, {n.y:.2f})")
print("walls stay at x = 0..10, y = 0..6")


In [ ]:
overhung_court = roof(square, 45.0, holes=[courtyard], overhang=0.5)
assert isinstance(overhung_court, Roof)
show("courtyard, 0.5 m overhang — outer eaves out, hole eaves in", overhung_court, square, [courtyard])

show(
    "overhang that closes the 4 m courtyard",
    roof(square, 45.0, holes=[courtyard], overhang=3.0),
    square,
    [courtyard],
)


## When the input cannot be roofed

`Failure` is a value. Each refusal is drawn as the input outline.


In [ ]:
cases = (
    ("out of range", square, 0.0, {}),
    ("bowtie", [(0.0, 0.0), (10.0, 10.0), (10.0, 0.0), (0.0, 10.0)], 45.0, {}),
    ("a line", [(0.0, 0.0), (10.0, 0.0), (4.0, 0.0)], 45.0, {}),
    (
        "parallel edges, two pitches",
        [(0.0, 0.0), (5.0, 0.0), (10.0, 0.0), (10.0, 6.0), (0.0, 6.0)],
        [45.0, 30.0, 45.0, 45.0, 45.0],
        {},
    ),
    (
        "hole touching the outer ring",
        square, 45.0,
        {"holes": [[(0.0, 0.0), (4.0, 0.0), (4.0, 4.0), (0.0, 4.0)]]},
    ),
)
for label, footprint, pitch, kwargs in cases:
    show(label, roof(footprint, pitch, **kwargs), footprint, kwargs.get("holes"))


## Validity, events, wavefront


In [ ]:
print(f"is_terrain: {rect_roof.validity.is_terrain}")
print(f"topology hash: {topology_hash(rect_roof)}")

logged = roof(l_shape, 45.0, events=True)
assert not isinstance(logged, Failure)
built, events = logged
print(f"{len(events)} events on the L-shape")
wavefront_view(rect_roof, 1.5, walls=rectangle).update_layout(
    title="rectangle wavefront at 1.5 m"
).show()
for fig in wavefront_steps(built, events, walls=l_shape):
    fig.show()


## What is here, and what is not

Convex, L, U, polygonal (including circular-ish) courtyards, per-edge
pitch, gables, overhang with walls drawn inside the eaves.

Not a chimney, dormer, or any other hole *through the slope*. Not a
true circular wall — only an n-gon. See `limitations.ipynb`.
